# 08 Selected Decoding Demo

This notebook inspects selected decoding behavior on held-out prompts:
- greedy token vs selected token
- candidate branches and scores
- repetition penalty effects

In [ ]:
from pathlib import Path
import sys

import torch
import yaml

NB_DIR = Path.cwd().resolve()
ROOT = NB_DIR.parents[2] if NB_DIR.name == "active" else Path.cwd().resolve()
SRC = ROOT / "GitHub" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from drift_selection.selected_decoding import SelectedDecodingConfig, compare_greedy_vs_selected_next_token
from drift_selection.transformer_pipeline import ensure_tokenizer_and_encoded_splits, load_main_config, load_prompt_bank
from drift_selection.transformer import load_model_config
from drift_selection.training import load_trained_model

cfg = load_main_config(ROOT, "GitHub/configs/transformer_teacher_student.yaml")
prep = ensure_tokenizer_and_encoded_splits(ROOT, cfg, force=False)
tok = prep["tokenizer"]


In [ ]:
out_root = ROOT / cfg["paths"]["root_outputs_dir"]
teacher_summary = yaml.safe_load((out_root / "teacher" / "training_summary.json").read_text())
teacher_ckpt = teacher_summary["history"]["best_checkpoint_path"] or teacher_summary["history"]["final_checkpoint_path"]
model_cfg = load_model_config(out_root / "teacher" / "model_config.json")
device = torch.device("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
teacher = load_trained_model(Path(teacher_ckpt), model_cfg, device)
sel_cfg_data = yaml.safe_load((ROOT / "GitHub/configs/selected_decoding.yaml").read_text())["selected_decoding"]
sel_cfg = SelectedDecodingConfig(**sel_cfg_data)

prompts = load_prompt_bank(out_root / "manifests" / "evaluation_prompts_pilot.json")
example = compare_greedy_vs_selected_next_token(teacher, prompts[0], sel_cfg, device)
example


In [ ]:
rows = example["candidate_table"][:8]
for i, row in enumerate(rows, 1):
    tok_id = row["first_token_id"]
    piece = tok.sp.id_to_piece(tok_id)
    score = row["branch_score"]
    branch = row["best_branch_ids"][:6]
    print(f"{i:02d}. token={piece!r} id={tok_id} score={score:.4f} best_branch={branch}")

print("Greedy token:", tok.sp.id_to_piece(example["greedy_next_token"]))
print("Selected token:", tok.sp.id_to_piece(example["selected_next_token"]))
